# Notebook 16: Preregistered Confirmatory C1–C3 Replication Study (Colab GPU)

**Protocol Identifier:** `PREREG-ELF-ARR2027-C1C3-REP20`  
**Governing Document:** `prereg/PREREGISTRATION.md` (OSF Timestamp Frozen)  
**Target Venue:** ACL 2027 (ARR Cycle)  
**Hardware Target:** Google Colab A100 GPU (~34.4 Compute Units for 120 runs)  

### Scientific Objectives:
1. Conduct the full-scale confirmatory replication of operating-point non-equivalence across 3 pairs (C1, C2, C3) and 20 fresh held-out seeds (401–420).
2. Generate $M=1,000$ passages per run across 120 total runs (120,000 passages total), evaluated under $K=32$ SDE steps ($NFE_{\mathrm{sample}} = 33$).
3. Enforce bit-identical paired Common Random Numbers (CRN) seeding across all three streams:
   - Initial latents $\mathbf{z}_0$ via `seed * 100003 + j`
   - Discretization schedules $t$ via `seed_int(seed, j, 'sched')`
   - SDE Brownian increments $\boldsymbol{\epsilon}_i$ via `seed_int(seed, j, 'sde')`
4. Parameter Accounting: ELF-B has 104,579,940 parameters (`n_params = sum(p.numel() for p in eval_model.parameters())`), strictly excluding the frozen T5-small encoder (~35M parameters). Model weights are loaded via `load_elf("ELF-B")` directly from HuggingFace Hub (`embedded-language-flows/ELF-B-owt-torch`).
5. Save raw generated passages (`samples.jsonl`) atomically to Google Drive for downstream consumption by NB17 (Evaluation Split) and NB15 (Automated LLM-Judge Panel).
6. Execute exact paired sign-flip permutation tests ($2^{20} = 1,048,576$ permutations) and hierarchical gatekeeping (Family 1 Joint TOST at $\alpha=0.05$ $\to$ Family 2 difference tests at $\alpha=0.05$).


In [ ]:
# Cell 1: Fast Google Colab Environment Setup, Drive Mount, & Dependency Check
import datetime
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import torch

# ==============================================================================
# EXECUTION CONFIGURATION FLAGS
# ==============================================================================
RUN_SMOKE_CPU     = False  # Tier 1: Fast local CPU synthetic unit test (<5s, for unittest discover)
RUN_PREFLIGHT_GPU = False  # Tier 2: Colab GPU "Tiny Real" preflight (~2 min on GPU, 100% real code)
# When both are False     # Tier 3: Full production Colab GPU execution
# ==============================================================================

class ColabEngineeringError(Exception):
    """Base exception for Colab engineering and reproducibility failures."""
    pass

class DriveNotMountedError(ColabEngineeringError):
    """Raised when Google Drive is not mounted at /content/drive."""
    pass

class MissingDriveArtifactError(ColabEngineeringError):
    """Raised when a required prerequisite file is missing on Drive."""
    pass

class PreregistrationNotFrozenError(ColabEngineeringError):
    """Raised when the OSF preregistration document is missing or modified."""
    pass

def assert_colab_drive_mounted() -> Path:
    if RUN_SMOKE_CPU:
        return Path.cwd()
    if "google.colab" not in sys.modules:
        raise ColabEngineeringError(
            "FAIL-CLOSED: This notebook must be executed inside a Google Colab GPU runtime.\n"
            "To test algorithmic logic locally on CPU, set RUN_SMOKE_CPU = True."
        )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/elf_research")
    if not root.is_dir():
        raise DriveNotMountedError(f"Drive mounted, but project root does not exist: {root}")
    return root

PROJECT_ROOT = assert_colab_drive_mounted()
print(f"[preflight] Operating directory: {PROJECT_ROOT} (RUN_SMOKE_CPU={RUN_SMOKE_CPU}, RUN_PREFLIGHT_GPU={RUN_PREFLIGHT_GPU})")

if not RUN_SMOKE_CPU:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [CODE_DIR, RUNS_DIR, REFS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    # 1. Unpack ELF/src and sync elfx to local Colab SSD (/content) to avoid Google Drive FUSE latency
    LOCAL_CODE_DIR = Path("/content/code") if Path("/content").is_dir() else CODE_DIR
    if torch.cuda.is_available():
        if not os.path.isdir("/content/repo/ELF/src"):
            zip_candidates = [CODE_DIR / "elf_code.zip", PROJECT_ROOT / "elf_code.zip", Path("/content/elf_code.zip")]
            found_zip = next((p for p in zip_candidates if p.exists()), None)
            if found_zip:
                print(f"[bootstrap] Unpacking ELF source from {found_zip} to /content/repo...")
                subprocess.run(["unzip", "-q", "-o", str(found_zip), "-d", "/content/repo"], check=True)
            else:
                print("[bootstrap] Cloning ELF repository into /content/repo/ELF (branch pytorch_elf)...")
                os.makedirs("/content/repo", exist_ok=True)
                subprocess.run([
                    "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                    "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
                ], check=True)

        if (CODE_DIR / "elfx").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
            shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)
        if (PROJECT_ROOT / "frontier_audit").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            shutil.copytree(PROJECT_ROOT / "frontier_audit", LOCAL_CODE_DIR / "frontier_audit", dirs_exist_ok=True)

    # 2. Configure environment variables and sys.path (prioritizing fast local SSD)
    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR), str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 3. Install only missing pip packages (avoids 60s pip resolver hang when packages are already installed)
    if torch.cuda.is_available():
        _REQ_SPECS = {"mauve": "mauve-text>=0.3.0", "sacrebleu": "sacrebleu>=2.4.0", "rouge_score": "rouge-score>=0.1.2", "einops": "einops>=0.7.0", "sentencepiece": "sentencepiece"}
        _missing_pkgs = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
        if _missing_pkgs:
            print(f"[bootstrap] Installing missing pip packages: {_missing_pkgs}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing_pkgs, check=False)

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        dev_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[bootstrap] Attached GPU: {dev_name} ({vram_gb:.1f} GiB VRAM)")

    print("[bootstrap] Environment ready.")
else:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print("[bootstrap] Local CPU testing mode enabled (RUN_SMOKE_CPU=True).")


In [ ]:
# 2. Paired Common Random Numbers (CRN) Seeding & Three-Stream Cryptographic Verification
def compute_tensor_digest(tensor: Any) -> str:
    """Computes SHA-256 cryptographic digest of contiguous array/tensor bytes."""
    if isinstance(tensor, torch.Tensor):
        arr = tensor.detach().cpu().contiguous().numpy()
    else:
        arr = np.ascontiguousarray(tensor)
    return hashlib.sha256(arr.tobytes()).hexdigest()

def generate_initial_latents(seed: int, sample_idx: int, seq_len: int = 256, dim: int = 768) -> np.ndarray:
    """Generates reproducible initial Gaussian latent vector z_0 using explicit seed arithmetic."""
    rng = np.random.RandomState(seed * 100003 + sample_idx)
    return rng.randn(seq_len, dim).astype(np.float32)

def verify_crn_pairing_cell(seed: int = 401, n_check: int = 2) -> Dict[str, Any]:
    """Verifies that Arm A and Arm B receive bit-identical initial latents under CRN (CPU test)."""
    digests_a = []
    digests_b = []
    for j in range(n_check):
        z_a = generate_initial_latents(seed=seed, sample_idx=j)
        z_b = generate_initial_latents(seed=seed, sample_idx=j)
        d_a = compute_tensor_digest(z_a)
        d_b = compute_tensor_digest(z_b)
        assert d_a == d_b, f"CRN latent mismatch at seed={seed}, sample={j}: {d_a} vs {d_b}"
        digests_a.append(d_a)
        digests_b.append(d_b)
    return {
        "status": "PASSED",
        "seed": seed,
        "samples_checked": n_check,
        "sample_digests": digests_a
    }

def verify_crn_three_stream_gpu(seed: int = 401, indices: Optional[List[int]] = None, device: str = "cuda") -> Dict[str, Any]:
    """
    Empirically audits all three CRN streams per seed in elfx.sample:
    1. Initial noise z_0: seed * 100003 + j (initial_noise)
    2. Discretization schedules t: seed_int(seed, j, 'sched') (sample_schedules)
    3. SDE Brownian increments: seed_int(seed, j, 'sde') (PerSampleNoise)
    Matches NB11 tensor digest audit.
    """
    from elfx import sample
    if indices is None:
        indices = [0, 1, 499, 999]
    L, D = 256, 768
    
    # 1. z0 stream
    z0_a = sample.initial_noise(indices, seed, L, D, noise_scale=1.0, device=device, dtype=torch.float32)
    z0_b = sample.initial_noise(indices, seed, L, D, noise_scale=1.0, device=device, dtype=torch.float32)
    sha_z0_a = compute_tensor_digest(z0_a)
    sha_z0_b = compute_tensor_digest(z0_b)
    assert sha_z0_a == sha_z0_b, f"CRN z0 digest mismatch at seed {seed}"
    
    # 2. Schedule stream
    sched_a = sample.sample_schedules(indices, seed, 32, "logit_normal", -0.6, 1.2, device, dtype=torch.float32)
    sched_b = sample.sample_schedules(indices, seed, 32, "logit_normal", -0.6, 1.2, device, dtype=torch.float32)
    sha_sched_a = compute_tensor_digest(sched_a)
    sha_sched_b = compute_tensor_digest(sched_b)
    assert sha_sched_a == sha_sched_b, f"CRN schedule digest mismatch at seed {seed}"
    
    # 3. Brownian increment stream
    eps_a_obj = sample.PerSampleNoise(indices, seed, (L, D), device, dtype=torch.float32)
    eps_b_obj = sample.PerSampleNoise(indices, seed, (L, D), device, dtype=torch.float32)
    eps_a = eps_a_obj(0)
    eps_b = eps_b_obj(0)
    sha_eps_a = compute_tensor_digest(eps_a)
    sha_eps_b = compute_tensor_digest(eps_b)
    assert sha_eps_a == sha_eps_b, f"CRN Brownian increment digest mismatch at seed {seed}"
    
    return {
        "status": "PASSED",
        "seed": seed,
        "indices_checked": indices,
        "sha_z0": sha_z0_a,
        "sha_sched": sha_sched_a,
        "sha_eps": sha_eps_a
    }

crn_status = verify_crn_pairing_cell(seed=401, n_check=2)
print(f"[crn_audit] Latent digest verification: {crn_status['status']} (SHA-256: {crn_status['sample_digests'][0][:16]}...)")


In [ ]:
# 3. Sampling Engine & Atomic Artifact Persistence
# Frozen operating points from prereg/PREREGISTRATION.md
FROZEN_OPERATING_POINTS = {
    "C1": {
        "arm_a": {"sc_cfg": 1.0, "gamma": 1.346},
        "arm_b": {"sc_cfg": 2.0, "gamma": 1.000},
        "target_ppl": 32.40,
        "target_h": 5.250,
        "ppl_margin": 32.40 * 0.025,  # +/- 0.81
        "h_margin": 0.015,
    },
    "C2": {
        "arm_a": {"sc_cfg": 1.0, "gamma": 1.547},
        "arm_b": {"sc_cfg": 3.0, "gamma": 1.000},
        "target_ppl": 28.90,
        "target_h": 5.210,
        "ppl_margin": 28.90 * 0.025,  # +/- 0.72
        "h_margin": 0.015,
    },
    "C3": {
        "arm_a": {"sc_cfg": 1.0, "gamma": 1.651},
        "arm_b": {"sc_cfg": 4.0, "gamma": 1.000},
        "target_ppl": 27.50,
        "target_h": 5.190,
        "ppl_margin": 27.50 * 0.025,  # +/- 0.69
        "h_margin": 0.015,
    },
}

REPLICATION_SEEDS = list(range(401, 421))  # 20 fresh preregistered seeds (401..420)

def execute_replication_run(
    pair_id: str,
    arm: str,
    seed: int,
    cfg: Dict[str, Any],
    project_root: Path,
    n_samples: int = 1000,
    steps: int = 32,
    device: str = "cuda"
) -> Dict[str, Any]:
    """
    Production execution engine with atomic checkpoint/resume safety.
    1. Skips run if summary.json already exists on disk.
    2. Reuses samples.jsonl if text generation completed but scoring was interrupted.
    3. Generates M=1000 samples via elfx.sample.generate and evaluates full metrics suite.
    """
    import elfx.load
    import elfx.sample
    import elfx.metrics
    import elfx.non_identifiability
    import elfx.refs
    import elfx.sweep

    gamma = float(cfg["gamma"])
    sc_cfg = float(cfg["sc_cfg"])
    method = "ode" if gamma == 0.0 else "sde"
    run_id = f"p4_elfb_s{int(steps)}_g{gamma}_sc{sc_cfg}_seed{int(seed)}"
    run_dir = project_root / "runs" / run_id
    summary_path = run_dir / "summary.json"
    samples_path = run_dir / "samples.jsonl"
    ext_path = run_dir / "extended_metrics.json"

    # 1. Skip if fully completed
    if summary_path.is_file() and ext_path.is_file() and samples_path.is_file():
        try:
            with open(summary_path, "r", encoding="utf-8") as f:
                res = json.load(f)
            with open(ext_path, "r", encoding="utf-8") as f:
                res.update(json.load(f))
            if (res.get("distinct2") is None or res.get("distinct2") == 0.0) and res.get("cross_distinct2") is not None:
                res["distinct2"] = float(res["cross_distinct2"])
            if "gen_ppl" in res and "rep4" in res and "jsd_pi" in res:
                print(f"[skip] {run_id} already exists with valid metrics. Reusing.")
                return res
        except Exception:
            pass

    run_dir.mkdir(parents=True, exist_ok=True)

    # 2. Text generation or sample reuse
    if samples_path.is_file():
        print(f"[reuse] {run_id} samples.jsonl exists. Proceeding directly to scoring.")
        texts = []
        with open(samples_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    texts.append(json.loads(line)["text"])
    else:
        print(f"[generate] Rolling out {run_id} (n={n_samples}, steps={steps}, method={method})...")
        t0 = time.time()
        model_dict = elfx.sweep.get_model("ELF-B")
        gen_out = elfx.sample.generate(
            model=model_dict["model"],
            config=model_dict["config"],
            tokenizer=model_dict["tokenizer"],
            n=n_samples,
            seed=seed,
            steps=steps,
            method=method,
            gamma=gamma,
            sc_cfg=sc_cfg,
            batch_size=64
        )
        texts = gen_out["texts"]
        gen_wall_s = time.time() - t0

        # Save samples.jsonl atomically
        tmp_samples = run_dir / "samples.tmp.jsonl"
        with open(tmp_samples, "w", encoding="utf-8") as f:
            for idx, text in enumerate(texts):
                f.write(json.dumps({"i": idx, "text": text}) + "\n")
        tmp_samples.replace(samples_path)

        if "token_ids" in gen_out and gen_out["token_ids"] is not None:
            t_ids = gen_out["token_ids"]
            if hasattr(t_ids, "cpu"):
                t_ids = t_ids.cpu().numpy()
            np.savez_compressed(run_dir / "tokens.npz", token_ids=np.asarray(t_ids))

    # 3. Autoregressive scoring with GPT-2 Large
    print(f"[score] Scoring {len(texts)} samples for {run_id}...")
    sc_res = elfx.metrics.score_gpt2(texts, batch_size=32)
    st_res = elfx.metrics.text_stats(texts)
    ppl = float(elfx.metrics.corpus_ppl(sc_res["nll"], sc_res["count"]))
    entropy = float(np.nanmean(sc_res["entropy"]))
    rep4 = float(st_res["rep4"])
    
    np.savez_compressed(
        run_dir / "per_sample.npz",
        nll=sc_res["nll"],
        count=sc_res["count"],
        entropy=sc_res["entropy"],
        empty=sc_res["empty"],
        rep4=st_res["per_sample_rep4"],
        degen=st_res["per_sample_degen"]
    )

    # 4. Secondary metrics (JSD_pi, Self-BLEU, CTW JCD)
    ref_texts = elfx.refs.get_owt_refs(1000)
    ext_metrics = elfx.non_identifiability.score_extended_sample_metrics(
        texts=texts,
        ref_texts=ref_texts,
        ctw_depth=6,
        seed=seed
    )
    with open(ext_path, "w", encoding="utf-8") as f:
        json.dump(ext_metrics, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

    # 5. MAUVE with gpt2-large features
    gen_feats = elfx.metrics.gpt2_features(texts, batch_size=16)
    ref_feats = elfx.metrics.gpt2_features(ref_texts, batch_size=16)
    mauve_val = float(elfx.metrics.mauve_from_features(gen_feats=gen_feats, ref_feats=ref_feats, seed=25))

    def _get_ext_metric(ext_d: Dict[str, Any], primary: str, *aliases: str) -> Optional[ float ]:
        for k in (primary,) + aliases:
            if k in ext_d and ext_d[k] is not None:
                val = float(ext_d[k])
                if math.isfinite(val):
                    return val
        print(f"[WARN] Missing metric '{primary}' (aliases={aliases}) in {run_id}; recording None.")
        return None

    summary = {
        "run_id": run_id,
        "pair": pair_id,
        "arm": arm,
        "seed": seed,
        "gamma": gamma,
        "sc_cfg": sc_cfg,
        "steps": steps,
        "n_samples": len(texts),
        "gen_ppl": ppl,
        "h_ps": entropy,
        "rep4": rep4,
        "jsd_pi": _get_ext_metric(ext_metrics, "jsd_inter_sample", "jsd_pi"),
        "self_bleu": _get_ext_metric(ext_metrics, "self_bleu"),
        "distinct2": _get_ext_metric(ext_metrics, "cross_distinct2", "distinct2", "distinct_2"),
        "jcd_bits_per_byte": _get_ext_metric(ext_metrics, "jcd_bits_per_byte"),
        "mauve": mauve_val,
        "status": "completed"
    }

    tmp_summary = run_dir / "summary.tmp.json"
    with open(tmp_summary, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    tmp_summary.replace(summary_path)

    summary.update(ext_metrics)
    return summary

In [ ]:
# 4. Exact Permutation Tests (2^20) & Hierarchical Gatekeeping
def exact_sign_flip_permutation_test(diffs: np.ndarray, chunk_size: int = 65536) -> float:
    """Computes exact two-sided p-value over all 2^20 = 1,048,576 sign permutations."""
    n = len(diffs)
    if n != 20:
        raise ValueError(f"Exact 2^20 sign-flip test requires n=20, got n={n}")
    
    obs_t = np.abs(np.mean(diffs))
    diffs_vec = diffs.astype(np.float32)
    total_perms = 1 << 20  # 1,048,576
    n_chunks = total_perms // chunk_size
    n_more_extreme = 0
    
    for chunk in range(n_chunks):
        sub_idx = np.arange(chunk * chunk_size, (chunk + 1) * chunk_size, dtype=np.int32)
        signs = np.empty((20, chunk_size), dtype=np.float32)
        for b in range(20):
            signs[b] = ((sub_idx >> b) & 1) * 2 - 1
        perm_means = (diffs_vec @ signs) / 20.0
        n_more_extreme += int(np.sum(np.abs(perm_means) >= (obs_t - 1e-12)))
        
    return float(n_more_extreme / total_perms)

def run_gatekeeping_analysis(run_records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Executes Family 1 gate (m=3 joint TOSTs, alpha=0.05) and Family 2 confirmatory testing (m=9 difference tests, alpha=0.05).
    Uses frontier_audit.tost.
    """
    try:
        import frontier_audit.tost as T
    except ImportError:
        import types
        from scipy import stats as _sp_stats
        class _TOSTRes:
            def __init__(self, diff, p_tost, p_two_sided_diff, is_equivalent):
                self.diff = diff
                self.p_tost = p_tost
                self.p_two_sided_diff = p_two_sided_diff
                self.is_equivalent = is_equivalent
            def to_dict(self):
                return self.__dict__
        def _tost_paired(a, b, low, high, alpha=0.05):
            d = np.asarray(b, dtype=np.float64) - np.asarray(a, dtype=np.float64)
            n = len(d)
            df = float(n - 1)
            md = float(np.mean(d))
            se = max(float(np.std(d, ddof=1)) / np.sqrt(n), 1e-15)
            p1 = float(_sp_stats.t.sf((md - low) / se, df=df))
            p2 = float(_sp_stats.t.sf((high - md) / se, df=df))
            p_tost = max(p1, p2)
            p_two = float(2.0 * _sp_stats.t.sf(abs(md / se), df=df))
            return _TOSTRes(md, p_tost, p_two, bool(p_tost < alpha))
        def _joint_tost(res_list):
            return {
                "p_joint": float(max(r.p_tost for r in res_list)),
                "is_equivalent": bool(all(r.is_equivalent for r in res_list)),
                "individual_results": [r.to_dict() for r in res_list],
            }
        def _holm_bonferroni(p_values, alpha=0.05):
            keys = list(p_values.keys())
            raw_p = np.array([p_values[k] for k in keys], dtype=np.float64)
            m = len(raw_p)
            sort_idx = np.argsort(raw_p)
            sorted_p = raw_p[sort_idx]
            adj_sorted = np.zeros(m, dtype=np.float64)
            run_max = 0.0
            for rank, p in enumerate(sorted_p):
                run_max = max(run_max, p * (m - rank))
                adj_sorted[rank] = min(1.0, run_max)
            adj_p = np.zeros(m, dtype=np.float64)
            adj_p[sort_idx] = adj_sorted
            return {
                "adjusted_p_values": {keys[i]: float(adj_p[i]) for i in range(m)},
                "significant": {keys[i]: bool(adj_p[i] < alpha) for i in range(m)},
                "m": m,
                "alpha": alpha,
            }
        def _hierarchical_gatekeeping(f1_p, f2_p, alpha=0.05):
            f1_h = _holm_bonferroni(f1_p, alpha=alpha)
            f1_pass = all(f1_h["significant"].values())
            if f1_pass:
                gate_status = "OPEN"
                f2_h = _holm_bonferroni(f2_p, alpha=alpha)
                f2_adj, f2_sig = f2_h["adjusted_p_values"], f2_h["significant"]
            else:
                gate_status = "CLOSED"
                f2_adj = {k: 1.0 for k in f2_p}
                f2_sig = {k: False for k in f2_p}
            return {
                "gate_status": gate_status,
                "family1": {
                    "name": "Family 1: Operating-Point Equivalence Gate",
                    "m": len(f1_p), "alpha": alpha, "raw_p_values": f1_p,
                    "adjusted_p_values": f1_h["adjusted_p_values"],
                    "significant": f1_h["significant"], "all_passed": f1_pass,
                },
                "family2": {
                    "name": "Family 2: Confirmatory Secondary Metrics",
                    "m": len(f2_p), "alpha": alpha, "raw_p_values": f2_p,
                    "adjusted_p_values": f2_adj, "significant": f2_sig,
                    "n_significant": sum(1 for v in f2_sig.values() if v),
                },
            }
        T = types.SimpleNamespace(
            tost_paired=_tost_paired,
            joint_tost=_joint_tost,
            holm_bonferroni=_holm_bonferroni,
            hierarchical_gatekeeping=_hierarchical_gatekeeping,
        )
    
    f1_p_values = {}
    family2_raw = {}
    details = {}
    
    for pair_id in ["C1", "C2", "C3"]:
        meta = FROZEN_OPERATING_POINTS[pair_id]
        a_records = [r for r in run_records if r["pair"] == pair_id and r["arm"] == "arm_a"]
        b_records = [r for r in run_records if r["pair"] == pair_id and r["arm"] == "arm_b"]
        a_records.sort(key=lambda x: x["seed"])
        b_records.sort(key=lambda x: x["seed"])
        
        ppl_a = [r["gen_ppl"] for r in a_records]
        ppl_b = [r["gen_ppl"] for r in b_records]
        h_a = [r["h_ps"] for r in a_records]
        h_b = [r["h_ps"] for r in b_records]
        
        res_ppl = T.tost_paired(ppl_a, ppl_b, -meta["ppl_margin"], meta["ppl_margin"])
        res_h = T.tost_paired(h_a, h_b, -meta["h_margin"], meta["h_margin"])
        joint = T.joint_tost([res_ppl, res_h])
        f1_p_values[pair_id] = joint["p_joint"]
        
        details[pair_id] = {
            "mean_ppl_a": float(np.mean(ppl_a)), "sd_ppl_a": float(np.std(ppl_a, ddof=1)),
            "mean_ppl_b": float(np.mean(ppl_b)), "sd_ppl_b": float(np.std(ppl_b, ddof=1)),
            "mean_h_a": float(np.mean(h_a)), "sd_h_a": float(np.std(h_a, ddof=1)),
            "mean_h_b": float(np.mean(h_b)), "sd_h_b": float(np.std(h_b, ddof=1)),
            "res_ppl": res_ppl.to_dict() if hasattr(res_ppl, "to_dict") else res_ppl,
            "res_h": res_h.to_dict() if hasattr(res_h, "to_dict") else res_h,
            "p_joint": float(joint["p_joint"]),
            "secondary": {},
        }
        
        # Family 2 secondary metrics: rep-4, JSD_pi, MAUVE (plus extended diagnostics)
        for metric in ["rep4", "jsd_pi", "mauve", "self_bleu", "distinct2", "jcd_bits_per_byte"]:
            if not all((metric in r and r[metric] is not None and math.isfinite(float(r[metric]))) for r in a_records + b_records):
                print(f"[WARN] Omitting metric '{metric}' on {pair_id}: missing or non-finite values detected.")
                continue
            val_a = np.array([r[metric] for r in a_records], dtype=np.float64)
            val_b = np.array([r[metric] for r in b_records], dtype=np.float64)
            diffs = val_b - val_a
            t_res = T.tost_paired(val_a, val_b, -0.01, 0.01)  # Two-sided paired diff test
            if metric in ["rep4", "jsd_pi", "mauve"]:
                family2_raw[f"{pair_id}_{metric}"] = float(t_res.p_two_sided_diff)
            p_perm = exact_sign_flip_permutation_test(diffs) if len(diffs) == 20 else float(t_res.p_two_sided_diff)
            details[pair_id]["secondary"][metric] = {
                "mean_a": float(np.mean(val_a)),
                "sd_a": float(np.std(val_a, ddof=1)),
                "mean_b": float(np.mean(val_b)),
                "sd_b": float(np.std(val_b, ddof=1)),
                "diff": float(np.mean(diffs)),
                "sd_diff": float(np.std(diffs, ddof=1)),
                "p_paired_t": float(t_res.p_two_sided_diff),
                "p_exact_perm_2_20": float(p_perm),
            }
            
    gate_results = T.hierarchical_gatekeeping(f1_p_values, family2_raw, alpha=0.05)
    gate_results["details"] = details
    return gate_results


In [ ]:
# 5. LaTeX Tabular Export & Manifest Generation
def export_replication_summary_table(gate_results: Dict[str, Any], output_path: Optional[Path] = None) -> str:
    """Formats LaTeX rows for Table 2 replication inclusion."""
    lines = [
        r"% Replication Results across Seeds 401-420",
        r"egin{tabular}{lcccc}",
        r"	oprule",
        r"Pair & Joint TOST ($p_{\mathrm{joint}}$) & Holm Gate & Family 2 Passed & Gate Status \\",
        r"\midrule"
    ]
    gate = gate_results["gate_status"]
    for pair_id in ["C1", "C2", "C3"]:
        raw_p = gate_results["family1"]["raw_p_values"][pair_id]
        adj_p = gate_results["family1"]["adjusted_p_values"][pair_id]
        sig = gate_results["family1"]["significant"][pair_id]
        lines.append(f"{pair_id} & {raw_p:.4f} & {adj_p:.4f} & {'Yes' if sig else 'No'} & {gate} \\\\")
    lines.append(r"ottomrule")
    lines.append(r"\end{tabular}")
    content = "\n".join(lines)
    if output_path:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(content, encoding="utf-8")
    return content

def build_replication_manifest(run_records: List[Dict[str, Any]], project_root: Path) -> Path:
    """Generates cryptographic SHA-256 manifest over all 120 completed runs."""
    rep_dir = project_root / "runs" / "replication_c1_c3"
    rep_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "n_runs": len(run_records),
        "seeds": REPLICATION_SEEDS,
        "runs": {}
    }
    for rec in run_records:
        rid = rec["run_id"]
        rd = project_root / "runs" / rid
        files_map = {}
        for fn in ["summary.json", "samples.jsonl", "extended_metrics.json", "per_sample.npz", "tokens.npz"]:
            fp = rd / fn
            if fp.is_file():
                files_map[fn] = hashlib.sha256(fp.read_bytes()).hexdigest()
        manifest["runs"][rid] = files_map
        
    manifest_path = rep_dir / "replication_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    return manifest_path


In [ ]:
# 6. Execution Router (CPU Smoke / GPU Preflight / Production Replication)
def run_smoke_cpu():
    """Runs end-to-end CPU verification of Notebook 16 algorithms."""
    print("[smoke] Running Notebook 16 CPU smoke test...")

    def mock_generate_and_save_run(run_dir: Path, pair_id: str, arm: str, seed: int, n_samples: int = 5) -> Dict[str, Any]:
        run_dir.mkdir(parents=True, exist_ok=True)
        samples_file = run_dir / "samples.jsonl"
        summary_file = run_dir / "summary.json"
        rng = np.random.RandomState(seed + (100 if arm == "arm_b" else 0))
        cfg = FROZEN_OPERATING_POINTS[pair_id][arm]
        with open(samples_file, "w", encoding="utf-8") as f:
            for idx in range(n_samples):
                rec = {
                    "sample_idx": idx,
                    "seed": seed,
                    "pair": pair_id,
                    "arm": arm,
                    "text": f"Replication text sample {idx} for pair {pair_id} {arm} on seed {seed}."
                }
                f.write(json.dumps(rec) + "\n")
        base_ppl = FROZEN_OPERATING_POINTS[pair_id]["target_ppl"]
        base_h = FROZEN_OPERATING_POINTS[pair_id]["target_h"]
        ppl = float(base_ppl + rng.normal(0.0, 0.15))
        h = float(base_h + rng.normal(0.0, 0.004))
        rep4 = float((0.15 if arm == "arm_b" else 0.08) + rng.normal(0.0, 0.005))
        jsd = float((0.28 if arm == "arm_b" else 0.22) + rng.normal(0.0, 0.005))
        mauve = float((0.92 if arm == "arm_b" else 0.88) + rng.normal(0.0, 0.005))
        summary = {
            "run_id": run_dir.name,
            "pair": pair_id,
            "arm": arm,
            "seed": seed,
            "n_samples": n_samples,
            "sc_cfg": cfg["sc_cfg"],
            "gamma": cfg["gamma"],
            "gen_ppl": ppl,
            "h_ps": h,
            "rep4": rep4,
            "jsd_pi": jsd,
            "mauve": mauve,
            "samples_path": str(samples_file.name)
        }
        with open(summary_file, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
        return summary

    # 1. CRN verification
    status = verify_crn_pairing_cell(seed=401, n_check=2)
    assert status["status"] == "PASSED"
    
    # 2. Mock 20-seed replication run records
    mock_records = []
    tmp_out = Path("/tmp/elf_smoke_nb16")
    for pair_id in ["C1", "C2", "C3"]:
        for arm in ["arm_a", "arm_b"]:
            for seed in REPLICATION_SEEDS:
                rec = mock_generate_and_save_run(tmp_out / f"{pair_id}_{arm}_s{seed}", pair_id, arm, seed, n_samples=5)
                mock_records.append(rec)
    
    # 3. Exact 2^20 permutation test
    sample_diffs = np.array([r["gen_ppl"] for r in mock_records[:20]]) - np.array([r["gen_ppl"] for r in mock_records[20:40]])
    p_perm = exact_sign_flip_permutation_test(sample_diffs)
    assert 0.0 <= p_perm <= 1.0, f"Permutation p-value invalid: {p_perm}"
    
    # 4. Gatekeeping analysis
    gate_res = run_gatekeeping_analysis(mock_records)
    assert "gate_status" in gate_res
    print(f"[smoke] Gate status: {gate_res['gate_status']}, Permutation test p: {p_perm:.6f}")
    print("[smoke] Notebook 16 CPU smoke test completed successfully.")

def run_preflight_gpu():
    """
    Tiny Real GPU Preflight Mode (~2 minutes on Colab A100):
    Executes the 100% REAL pipeline on 1 seed (401), M=8 samples, K=4 steps.
    Validates model download, CUDA generation, and GPT-2 Large scoring without burning credits.
    """
    print("="*70)
    print("[preflight-gpu] Launching Tiny Real Colab GPU Preflight on ELF-B...")
    if not torch.cuda.is_available():
        raise ColabEngineeringError("FAIL-CLOSED: GPU Preflight requires an attached CUDA GPU.")
    import elfx.load
    import elfx.sample
    import elfx.metrics
    import elfx.sweep
    
    t0 = time.time()
    # 1. Model Loading
    print("[preflight-gpu] Loading real ELF-B model via HuggingFace Hub...")
    model_dict = elfx.sweep.get_model("ELF-B")
    n_params = sum(p.numel() for p in model_dict["model"].parameters())
    print(f"[preflight-gpu] Loaded ELF-B: {n_params:,} parameters (excludes frozen T5-small encoder).")
    
    # 2. Three-Stream CRN Verification on GPU
    print("[preflight-gpu] Auditing three-stream CRN tensor digests on GPU...")
    crn_audit = verify_crn_three_stream_gpu(seed=401, indices=[0, 1, 7], device="cuda")
    print(f"[preflight-gpu] CRN audit: {crn_audit['status']} (z0 digest: {crn_audit['sha_z0'][:16]}...)")
    
    # 3. Tiny Real Rollout (M=8, K=4)
    print("[preflight-gpu] Rolling out M=8 sequences at K=4 steps (seed 401, Pair C1 Arm A)...\n")
    gen_out = elfx.sample.generate(
        model=model_dict["model"],
        config=model_dict["config"],
        tokenizer=model_dict["tokenizer"],
        n=8,
        seed=401,
        steps=4,
        method="sde",
        gamma=1.346,
        sc_cfg=1.0,
        batch_size=8
    )
    texts = gen_out["texts"]
    assert len(texts) == 8, f"Expected 8 texts, got {len(texts)}"
    print(f"[preflight-gpu] Generated 8 samples in {time.time() - t0:.2f}s. Sample preview: {texts[0][:80]!r}...")
    
    # 4. Real Autoregressive Scoring
    print("[preflight-gpu] Scoring generated samples with gpt2-large (bf16)...")
    sc = elfx.metrics.score_gpt2(texts, batch_size=4)
    ppl = elfx.metrics.corpus_ppl(sc["nll"], sc["count"])
    assert ppl > 1.0, f"Invalid perplexity: {ppl}"
    print(f"[preflight-gpu] Verified Gen-PPL = {ppl:.2f}. Pipeline fully functional.")
    print(f"[preflight-gpu] Completed Tiny Real GPU Preflight in {time.time() - t0:.1f}s. GPU ready for full run!")
    print("="*70)

def run_production_replication():
    """Full Production Replicated Experiment (120 runs across C1..C3 and seeds 401..420)."""
    if not torch.cuda.is_available():
        raise ColabEngineeringError("FAIL-CLOSED: Production replication requires a CUDA GPU.")
    print("="*70)
    print("[production] Beginning full 120-run confirmatory replication on Colab GPU...")
    print("="*70)
    
    # 1. CRN Verification on GPU for all seeds
    print("[production] Auditing three-stream CRN digests across all 20 seeds...")
    for s in REPLICATION_SEEDS:
        audit = verify_crn_three_stream_gpu(seed=s, indices=[0, 1, 499, 999], device="cuda")
        assert audit["status"] == "PASSED"
    print("[production] All 20 replication seeds passed 3-stream CRN verification.")
    
    # 2. Execution Loop across 3 Pairs x 2 Arms x 20 Seeds = 120 Runs
    all_records = []
    total_runs = len(FROZEN_OPERATING_POINTS) * 2 * len(REPLICATION_SEEDS)
    run_idx = 0
    t_start = time.time()
    
    for pair_id in ["C1", "C2", "C3"]:
        pair_cfg = FROZEN_OPERATING_POINTS[pair_id]
        for arm in ["arm_a", "arm_b"]:
            arm_cfg = pair_cfg[arm]
            for seed in REPLICATION_SEEDS:
                run_idx += 1
                print(f"\n[{run_idx}/{total_runs}] Running {pair_id} {arm} (seed {seed})...")
                rec = execute_replication_run(
                    pair_id=pair_id,
                    arm=arm,
                    seed=seed,
                    cfg=arm_cfg,
                    project_root=PROJECT_ROOT,
                    n_samples=1000,
                    steps=32,
                    device="cuda"
                )
                all_records.append(rec)
                elapsed = time.time() - t_start
                avg_time = elapsed / run_idx
                rem_time = avg_time * (total_runs - run_idx)
                print(f"  -> Done {rec['run_id']}: Gen-PPL={rec['gen_ppl']:.2f}, H_ps={rec['h_ps']:.4f}, rep-4={rec['rep4']:.4f}, MAUVE={rec['mauve']:.3f} (ETA: {rem_time/60:.1f}m)")
                
    # 3. Gatekeeping & Exact Permutation Analysis
    print("\n[production] Running preregistered hierarchical gatekeeping analysis...")
    gate_results = run_gatekeeping_analysis(all_records)
    print(f"[production] Family 1 Gate Status: {gate_results['gate_status']}")
    
    # 4. Export LaTeX Table and SHA-256 Manifest
    rep_summary_path = PROJECT_ROOT / "runs" / "replication_c1_c3" / "replication_c1_c3_summary.json"
    rep_summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(rep_summary_path, "w", encoding="utf-8") as f:
        json.dump(gate_results, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
        
    tab_path = PROJECT_ROOT / "runs" / "replication_c1_c3" / "replication_table2_rows.tex"
    latex_tab = export_replication_summary_table(gate_results, tab_path)
    print(f"[production] Saved LaTeX summary table to {tab_path}")
    
    manifest_path = build_replication_manifest(all_records, PROJECT_ROOT)
    print(f"[production] Built SHA-256 manifest over 120 runs at {manifest_path}")
    print("[production] Confirmatory replication execution completed successfully!")

if RUN_SMOKE_CPU:
    run_smoke_cpu()
elif RUN_PREFLIGHT_GPU:
    run_preflight_gpu()
else:
    run_production_replication()